# 02 — Model Training & Evaluation

**SDI Tehran Pipeline · Notebook 2 of 5**

## Purpose
Train categorical classifiers and an MLP regressor (continuous attributes) on formal Tehran
segments. Serialise all models, scalers, and label encoders for use in notebook 03. Save all
performance scores to `model_scores.json`.

**Branch note (`experiment/class-weighting`):** categorical classifiers were switched from a
single `MLPClassifier` per attribute to two candidate models — `RandomForestClassifier` and
`HistGradientBoostingClassifier` — trained side by side with explicit `sample_weight`
(`compute_sample_weight('balanced', ...)`) to test whether class-imbalance weighting alone
improves the `landuse` classifier (Hypothesis 1, see phase 7a handoff). `policy_era` and
`dilap_index` are included too so weighting effects are visible across all three targets, not
just the one that motivated the experiment.

## Inputs
- `data/training_set.csv` — formal Tehran segments with surveyed labels and PC1/PC2/PC3 (written by NB01)

## Outputs
- `trained_models/clf_{attr}_random_forest.joblib`, `clf_{attr}_hist_gb.joblib` — per-attribute, per-candidate classifiers
- `trained_models/le_{attr}.joblib` — per-attribute LabelEncoder
- `trained_models/regressor.joblib` — multi-output MLPRegressor
- `models/model_scaler.joblib` — StandardScaler fitted on PC1/PC2/PC3 training set
- `data/outputs/model_scores.json` — performance metrics (baseline Cramér's V + R², unchanged pending experiment outcome)
- `evaluation_reports/` — per-attribute, per-model classification reports
- `evaluation_reports/categorical_model_comparison.json` — consolidated side-by-side comparison


## 0 · Config & imports

In [ ]:
import sys, json, os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, r2_score, mean_squared_error
)
import joblib

sys.path.insert(0, '..')
from config import (
    TRAINING_SET_CSV, MODELS_DIR, TRAINED_MODELS_DIR,
    MODEL_SCORES_JSON, EVAL_DIR,
    MODEL_SCALER_PATH,
    CATEGORICAL_TARGETS, CONTINUOUS_TARGETS,
    MLP_HIDDEN_LAYERS, MLP_MAX_ITER, RANDOM_STATE, TEST_SIZE,
    RF_N_ESTIMATORS, RF_MAX_DEPTH,
    HGB_MAX_ITER, HGB_LEARNING_RATE, HGB_MAX_DEPTH,
    SEG_ID_COL
)

# Ensure output directories exist
for d in [MODELS_DIR, TRAINED_MODELS_DIR, EVAL_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Config loaded.")
print(f"  Categorical targets : {CATEGORICAL_TARGETS}")
print(f"  Continuous targets  : {CONTINUOUS_TARGETS}")

Config loaded.
  Categorical targets : ['policy_era', 'dilap_index', 'landuse']
  Continuous targets  : ['mean_area', 'perimeter', 'mean_pop']


## 1 · Load training set

PC1/PC2/PC3 were appended to `training_set.csv` by NB01 — no join required.

In [ ]:
df = pd.read_csv(TRAINING_SET_CSV)
print(f"Training set loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")

# Verify PC columns are present
for col in ['PC1', 'PC2', 'PC3']:
    assert col in df.columns, f"'{col}' missing — re-run NB01 first."

missing = df[CATEGORICAL_TARGETS + CONTINUOUS_TARGETS].isnull().sum()
print("\nNull counts per target attribute:")
print(missing[missing > 0] if missing.any() else "  None")

Training set loaded: 147,414 rows
Columns: ['Depthmap_R', 'Angular_Co', 'Connectivi', 'Segment_Le', 'mean_area', 'perimeter', 'landuse', 'mean_pop', 'dilap_index', 'policy_era', 'CH_10k', 'CH_1200', 'CH_2000', 'CH_20k', 'CH_400', 'CH_40k', 'CH_5000', 'CH_800', 'INT_10k', 'INT_1200', 'INT_2000', 'INT_20k', 'INT_400', 'INT_40k', 'INT_5000', 'INT_800', 'NC_10k', 'NC_1200', 'NC_2000', 'NC_20k', 'NC_400', 'NC_40k', 'NC_5000', 'NC_800', 'TD_10k', 'TD_1200', 'TD_2000', 'TD_20k', 'TD_400', 'TD_40k', 'TD_5000', 'TD_800', 'PC1', 'PC2', 'PC3']

Null counts per target attribute:
dilap_index     5816
landuse        48928
mean_area      48928
perimeter      48928
mean_pop        6029
dtype: int64


## 2 · Prepare features

Model input is [PC1, PC2, PC3]. A `StandardScaler` is fitted here on the training set only and saved as `model_scaler.joblib`. NB03 loads this scaler without refitting, ensuring the periphery segments are transformed into the same feature space.

In [ ]:
X_raw = df[['PC1', 'PC2', 'PC3']].values

model_scaler = StandardScaler()
X = model_scaler.fit_transform(X_raw)

joblib.dump(model_scaler, MODEL_SCALER_PATH)
print(f"Saved model scaler: {MODEL_SCALER_PATH}")
print(f"Feature matrix shape: {X.shape}")

Saved model scaler: D:\SID\models\model_scaler.joblib
Feature matrix shape: (147414, 3)


## 3 · Train classifiers (categorical attributes)

**Methodology note (`experiment/class-weighting`):** for each categorical attribute, two
candidate models — `RandomForestClassifier` and `HistGradientBoostingClassifier` — are trained
side by side on the same train/test split, both using the same explicit `sample_weight` computed
via `compute_sample_weight('balanced', y_train)`. Using an explicit weight vector (rather than
each model's own `class_weight='balanced'` shortcut) keeps the weighting scheme identical across
both candidates, and works uniformly under this project's pinned `scikit-learn<1.3`, where
`HistGradientBoostingClassifier` does not yet expose a native `class_weight` argument.

The prior MLPClassifier's Cramér's V scores (established in earlier analysis) are kept as a
fixed baseline reference, not recomputed — this experiment evaluates candidate replacements
against that baseline, it does not overwrite it. `model_scores.json` (Section 5) is left
pointing at the baseline scores until a decision is made on adoption.


In [ ]:
# Baseline reference: Cramer's V scores from the original per-attribute MLPClassifier
# (documented in project notes) - fixed comparison point, not recomputed here.
BASELINE_MLP_CRAMERS_V = {
    "policy_era":  0.3460,
    "dilap_index": 0.2507,
    "landuse":     0.1058,
}

CANDIDATE_MODELS = {
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "hist_gb": lambda: HistGradientBoostingClassifier(
        max_iter=HGB_MAX_ITER,
        learning_rate=HGB_LEARNING_RATE,
        max_depth=HGB_MAX_DEPTH,
        random_state=RANDOM_STATE,
    ),
}

classifier_reports = {}   # attr -> {model_name: {...}, 'baseline_mlp_cramers_v': float}

for attr in CATEGORICAL_TARGETS:
    print(f"\n{'='*60}\n  Attribute: {attr}\n{'='*60}")

    mask = df[attr].notna()
    y_raw = df.loc[mask, attr]
    X_attr = X[mask]

    le = LabelEncoder()
    y = le.fit_transform(y_raw.values.ravel())
    class_names = [str(c) for c in le.classes_]
    print(f"  Classes: {class_names}")

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_attr, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # Computed once per attribute and reused across both candidates, so the
    # comparison isolates model family rather than weighting scheme.
    # 'balanced' reweights inversely to class frequency in y_tr.
    sample_weight = compute_sample_weight('balanced', y_tr)

    attr_results = {}

    for model_name, model_factory in CANDIDATE_MODELS.items():
        print(f"\n  --- {model_name} ---")
        clf = model_factory()
        clf.fit(X_tr, y_tr, sample_weight=sample_weight)
        y_pred = clf.predict(X_te)

        acc         = accuracy_score(y_te, y_pred)
        f1_macro    = f1_score(y_te, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_te, y_pred, average='weighted', zero_division=0)
        report_dict = classification_report(
            y_te, y_pred, target_names=class_names, zero_division=0, output_dict=True
        )
        report_text = classification_report(
            y_te, y_pred, target_names=class_names, zero_division=0
        )

        print(f"    Accuracy    : {acc:.4f}")
        print(f"    Macro F1    : {f1_macro:.4f}")
        print(f"    Weighted F1 : {f1_weighted:.4f}")
        print(report_text)

        clf_path = os.path.join(TRAINED_MODELS_DIR, f'clf_{attr}_{model_name}.joblib')
        joblib.dump(clf, clf_path)
        print(f"    Saved: {clf_path}")

        report_path = os.path.join(EVAL_DIR, f'{attr}_{model_name}_report.txt')
        with open(report_path, 'w') as f:
            f.write(f"Attribute : {attr}\n")
            f.write(f"Model     : {model_name}\n")
            f.write(f"Accuracy  : {acc:.4f}\n")
            f.write(f"Macro F1  : {f1_macro:.4f}\n")
            f.write(f"Weighted F1: {f1_weighted:.4f}\n")
            f.write(f"(Baseline MLP Cramer's V, prior analysis: {BASELINE_MLP_CRAMERS_V[attr]:.4f})\n\n")
            f.write(report_text)
        print(f"    Saved: {report_path}")

        attr_results[model_name] = {
            'accuracy': acc,
            'macro_f1': f1_macro,
            'weighted_f1': f1_weighted,
            'per_class': report_dict,
        }

    le_path = os.path.join(TRAINED_MODELS_DIR, f'le_{attr}.joblib')
    joblib.dump(le, le_path)
    print(f"\n  Saved: {le_path}")

    attr_results['baseline_mlp_cramers_v'] = BASELINE_MLP_CRAMERS_V[attr]
    classifier_reports[attr] = attr_results


  Training classifier: policy_era
  Classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  Accuracy  : 0.3712
  Macro F1  : 0.2049
  Cramér's V: 0.3460  (from prior analysis)
              precision    recall  f1-score   support

           1       0.67      0.00      0.01       575
           2       0.40      0.05      0.09      1841
           3       0.41      0.07      0.12      2777
           4       0.39      0.71      0.50     10220
           5       0.34      0.09      0.14      7373
           6       0.34      0.41      0.37      6697

    accuracy                           0.37     29483
   macro avg       0.42      0.22      0.20     29483
weighted avg       0.37      0.37      0.31     29483

  Saved: D:\SID\trained_models\clf_policy_era.joblib
  Saved: D:\SID\trained_models\le_policy_era.joblib

  Training classifier: dilap_index
  Classes: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float

## 4 · Train regressor (continuous attributes)

**Methodology note:** A single multi-output MLPRegressor predicts `mean_area`, `perimeter`, and `mean_pop` simultaneously. R² scores are expected to be low — a known limitation of PCA-compressed features as model input. The weighting scheme in NB04 down-weights these attributes proportionally via their R² scores. `mean_area` may produce a negative R², meaning the model performs worse than a mean baseline; this is documented as a methodological note in the paper.

In [ ]:
print(f"\n{'='*50}\n  Training regressor: {CONTINUOUS_TARGETS}\n{'='*50}")

mask_cont = df[CONTINUOUS_TARGETS].notna().all(axis=1)
y_cont = df.loc[mask_cont, CONTINUOUS_TARGETS].values
X_cont = X[mask_cont]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cont, y_cont, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

reg = MLPRegressor(
    hidden_layer_sizes=MLP_HIDDEN_LAYERS,
    max_iter=MLP_MAX_ITER,
    random_state=RANDOM_STATE
)
reg.fit(X_tr, y_tr)
y_pred = reg.predict(X_te)

r2_scores  = {}
mse_scores = {}
for i, attr in enumerate(CONTINUOUS_TARGETS):
    r2  = r2_score(y_te[:, i], y_pred[:, i])
    mse = mean_squared_error(y_te[:, i], y_pred[:, i])
    r2_scores[attr]  = r2
    mse_scores[attr] = mse
    print(f"  {attr:<15}  R²: {r2:.4f}  MSE: {mse:.4f}")

reg_path = os.path.join(TRAINED_MODELS_DIR, 'regressor.joblib')
joblib.dump(reg, reg_path)
print(f"\n  Saved: {reg_path}")


  Training regressor: ['mean_area', 'perimeter', 'mean_pop']
  mean_area        R²: -0.0001  MSE: 4320050665.2910
  perimeter        R²: 0.0049  MSE: 46619.6768
  mean_pop         R²: -0.0000  MSE: 231231.6513

  Saved: D:\SID\trained_models\regressor.joblib


## 5 · Compile and save model_scores.json

Categorical attributes are weighted by Cramér's V; continuous by R². These scores flow into
NB04's SDI weighting scheme.

**Branch note:** this file is left unchanged from the baseline MLP scores on
`experiment/class-weighting` — the RF/HGB candidates are evaluation-only here (see
`categorical_model_comparison.json` in Section 6). Update this file only once a candidate is
formally adopted into `main`.


In [ ]:
model_scores = {
    'policy_era':  {'metric': 'cramers_v', 'score': BASELINE_MLP_CRAMERS_V['policy_era']},
    'dilap_index': {'metric': 'cramers_v', 'score': BASELINE_MLP_CRAMERS_V['dilap_index']},
    'landuse':     {'metric': 'cramers_v', 'score': BASELINE_MLP_CRAMERS_V['landuse']},
    'mean_area':   {'metric': 'r2',        'score': r2_scores['mean_area']},
    'perimeter':   {'metric': 'r2',        'score': r2_scores['perimeter']},
    'mean_pop':    {'metric': 'r2',        'score': r2_scores['mean_pop']},
}

with open(MODEL_SCORES_JSON, 'w') as f:
    json.dump(model_scores, f, indent=2)

print(f"Saved: {MODEL_SCORES_JSON}")
print(json.dumps(model_scores, indent=2))


Saved: D:\SID\data\model_scores.json
{
  "policy_era": {
    "metric": "cramers_v",
    "score": 0.346
  },
  "dilap_index": {
    "metric": "cramers_v",
    "score": 0.2507
  },
  "landuse": {
    "metric": "cramers_v",
    "score": 0.1058
  },
  "mean_area": {
    "metric": "r2",
    "score": -0.00013765413987143482
  },
  "perimeter": {
    "metric": "r2",
    "score": 0.004895614356078659
  },
  "mean_pop": {
    "metric": "r2",
    "score": -4.765688311025862e-05
  }
}


## 6 · Consolidated comparison summary

Per-model, per-attribute `.txt` reports were already written in Section 3. This cell prints a
side-by-side summary table and saves the full comparison (including per-class precision/recall)
to a single JSON file for easy review in the PR.


In [ ]:
print(f"{'Attribute':<14}{'Model':<16}{'Accuracy':>10}{'Macro F1':>11}{'Weighted F1':>13}")
print("-" * 64)
for attr, results in classifier_reports.items():
    baseline = results['baseline_mlp_cramers_v']
    for model_name in CANDIDATE_MODELS.keys():
        info = results[model_name]
        print(f"{attr:<14}{model_name:<16}{info['accuracy']:>10.4f}{info['macro_f1']:>11.4f}{info['weighted_f1']:>13.4f}")
    print(f"{'':<14}{'(baseline MLP, Cramer V)':<16}{'':>10}{'':>11}{baseline:>13.4f}")
    print("-" * 64)

comparison_path = os.path.join(EVAL_DIR, 'categorical_model_comparison.json')
with open(comparison_path, 'w') as f:
    json.dump(classifier_reports, f, indent=2)
print(f"\nSaved consolidated comparison: {comparison_path}")


Saved: D:\SID\evaluation_reports\policy_era_report.txt
Saved: D:\SID\evaluation_reports\dilap_index_report.txt
Saved: D:\SID\evaluation_reports\landuse_report.txt
